# Read RMF table

In [1]:
import pandas as pd
import numpy as np
RFM = pd.read_csv('RMF_table.csv')

In the following section, for the purpose of segmentation [Champions, Loyal, Big Spenders, At Risk, Lost], I give a score to each customer for each RMF feature based on quantile analysis. `RMF_score = 532` means the customer got `R_score = 5`, `M_score = 3`, and `Frequency_score = 2`. For the Frequency and Recency features, since their values are discrete, I ranked all data entries. Customers with identical values received the same average rank, ensuring that tied values are treated equally. Imagine that you have 100 data entries with `frequency=1`. If you use quantile-based scoring, you would give different `frequency_score` to different customers, although they have the same value. So that is why I came up with this approach of ranking.

In [ ]:
# RFM['R_score'] = pd.qcut(RFM['Recency_Days'], 5, labels=[5,4,3,2,1])
rec_pct = RFM['Recency_Days'].rank(method='average', pct=True)
RFM['R_score'] = (6 - np.ceil(rec_pct*5)).astype(int).clip(1,5)


# RFM['Frequency_score'] = pd.qcut(RFM['Frequency'].rank(method = 'first'), 5, labels=[1,2,3,4,5])
freq_pct = RFM['Frequency'].rank(method = 'average', pct = True)
RFM['F_score'] = np.ceil(freq_pct * 5).astype(int).clip(1,5)

RFM['M_score'] = pd.qcut(RFM['Monetary'], 5, labels=[1,2,3,4,5])

RFM['RFM_score'] = RFM['R_score'].astype(str) + RFM['F_score'].astype(str) + RFM['M_score'].astype(str)
display(RFM.head())

,CustomerID,Recency_Days,Frequency,Monetary,R_score,Frequency_score,Monetary_score,RFM_score
0,12346.0,164.0,11,372.86,2,5,2,252
1,12347.0,2.0,2,1323.32,5,3,4,534
2,12348.0,73.0,1,222.16,2,1,1,211
3,12349.0,42.0,3,2671.14,3,3,5,335
4,12351.0,10.0,1,300.93,5,1,2,512


Now we can segment customers based on the RFM_score.

In [6]:
def segment_customer(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'
    elif row['F_score'] >= 4 and row['R_score'] >= 3:
        return 'Loyal'
    elif row['M_score'] >= 4:
        return 'Big Spenders'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Lost'
    else:
        return 'Need Attention'

RFM['Segment'] = RFM.apply(segment_customer, axis=1)
display(RFM.head())

KeyError: 'F_score'